# Analyse e-commerce avec PySpark — Dataset Olist

Ce notebook rassemble une série d'exercices PySpark réalisés sur le dataset public
**Brazilian E-Commerce (Olist)** : environ 100 000 commandes passées entre 2016 et 2018
sur une marketplace brésilienne, réparties dans 9 tables (clients, commandes, articles,
paiements, avis, produits, vendeurs, géolocalisation, traduction des catégories).

Les exercices suivent une progression en trois niveaux :

| Partie | Niveau | Thèmes |
|---|---|---|
| [ 1](#1) | Débutant | Contrôles de qualité, filtres, typage, colonnes calculées, fonctions de fenêtre |
| [ 2](#2) | Intermédiaire | Jointures, agrégations, table de faits vendeurs, broadcast join, écriture Parquet |
| [ 3](#3) | Avancé | Analyse client (réachat, Pareto 80/20), pivot, partitionnement, détection d'outliers, export JSON |

**Environnement :** Google Colab, PySpark en mode local.

> Chaque question suit la même démarche : explorer la donnée, appliquer la transformation,
> puis **contrôler le résultat** (comptages, sommes, cas limites) avant de passer à la suite.

## 0. Mise en place de l'environnement

Installation des dépendances, récupération du dataset depuis Kaggle et création de la session Spark.

In [ ]:
# Installation de PySpark (mode silencieux)
!pip install pyspark -q

In [ ]:
# Client Kaggle pour télécharger le dataset
!pip install -q kaggle

In [ ]:
# Authentification Kaggle : upload du fichier kaggle.json (clé API personnelle)
# puis copie dans ~/.kaggle avec les droits restreints exigés par le client.
# Remarque : le chemin testé est un chemin Windows local ; dans Colab il n'existe pas,
# donc l'upload est toujours demandé. Ne jamais versionner kaggle.json.
from google.colab import files
import os

if not os.path.exists('C:/Users/DELL5440/Downloads/kaggle.json'):
    print("Veuillez uploader votre fichier kaggle.json")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

Veuillez uploader votre fichier kaggle.json


Saving kaggle.json to kaggle.json


In [ ]:
# Téléchargement et décompression du dataset Olist dans le dossier olist_data/
!kaggle datasets download -d olistbr/brazilian-ecommerce
!unzip -o brazilian-ecommerce.zip -d olist_data

Dataset URL: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
License(s): CC-BY-NC-SA-4.0
100% 42.6M/42.6M [00:00<00:00, 60.0MB/s]

Archive:  brazilian-ecommerce.zip
  inflating: olist_data/olist_customers_dataset.csv  
  inflating: olist_data/olist_geolocation_dataset.csv  
  inflating: olist_data/olist_order_items_dataset.csv  
  inflating: olist_data/olist_order_payments_dataset.csv  
  inflating: olist_data/olist_order_reviews_dataset.csv  
  inflating: olist_data/olist_orders_dataset.csv  
  inflating: olist_data/olist_products_dataset.csv  
  inflating: olist_data/olist_sellers_dataset.csv  
  inflating: olist_data/product_category_name_translation.csv  


### Session Spark et chargement des données

In [ ]:
# Session Spark locale.
# shuffle.partitions réduit à 8 (200 par défaut) : plus adapté à un petit volume sur Colab.
# L'import * de pyspark.sql.functions remplace volontairement sum, min, max, round...
# par leurs équivalents Spark dans tout le notebook.
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("Olist_Intensive_Training") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

In [ ]:
# Chargement des 9 CSV dans un dictionnaire de DataFrames.
# Le nom de chaque table est simplifié : "olist_orders_dataset.csv" -> "orders".
datasets = [
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv"
]
dfs = {}
for dataset in datasets:
    name = dataset.replace(".csv", "").replace("olist_", "").replace("_dataset", "")
    path = f"olist_data/{dataset}"
    dfs[name] = spark.read.csv(path, header=True, inferSchema=True)
    print(f"Dataset '{name}' chargé. Lignes : {dfs[name].count()}")

Dataset 'customers' chargé. Lignes : 99441
Dataset 'geolocation' chargé. Lignes : 1000163
Dataset 'order_items' chargé. Lignes : 112650
Dataset 'order_payments' chargé. Lignes : 103886
Dataset 'order_reviews' chargé. Lignes : 104162
Dataset 'orders' chargé. Lignes : 99441
Dataset 'products' chargé. Lignes : 32951
Dataset 'sellers' chargé. Lignes : 3095
Dataset 'product_category_name_translation' chargé. Lignes : 71


In [ ]:
# Vérification des schémas inférés (types détectés par inferSchema)
for name, df in dfs.items():
    print(f"Schema pour le DataFrame : {name}")
    df.printSchema()
    print("-" * 40)

Schema pour le DataFrame : customers
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

----------------------------------------
Schema pour le DataFrame : geolocation
root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)

----------------------------------------
Schema pour le DataFrame : order_items
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullab

---
<a id="exercice-1"></a>
## Partie 1 — Niveau débutant : exploration et nettoyage

Objectif : prendre en main les tables, vérifier leur qualité et préparer les données
(filtres, gestion des valeurs nulles, typage, colonnes dérivées).

In [ ]:
# Raccourcis : une variable par table pour alléger l'écriture des exercices
customers = dfs['customers']
geolocation = dfs['geolocation']
order_items = dfs['order_items']
order_payments = dfs['order_payments']
order_reviews = dfs['order_reviews']
orders = dfs['orders']
products = dfs['products']
sellers = dfs['sellers']
category_translation = dfs['product_category_name_translation']

### Q0 — Unicité des identifiants clients

Vérifier si `customer_id` et `customer_unique_id` sont des identifiants uniques.

In [ ]:
# Un identifiant est unique si son nombre de valeurs distinctes = nombre de lignes.
# customer_id est lié à une commande ; customer_unique_id représente le client réel,
# qui peut donc apparaître plusieurs fois s'il a commandé plusieurs fois.
nbre_lignes = customers.count()
nbre_distinct_cust_id =customers.select('customer_id').distinct().count()
nbre_distinct_cust_unique_id =customers.select('customer_unique_id').distinct().count()
print("Lignes :", nbre_lignes)
print("customer_id distincts :", nbre_distinct_cust_id)
print("customer_unique_id distincts :", nbre_distinct_cust_unique_id)

print("customer_id unique ?", nbre_lignes == nbre_distinct_cust_id)
print("customer_unique_id unique ?", nbre_lignes == nbre_distinct_cust_unique_id)


Lignes : 99441
customer_id distincts : 99441
customer_unique_id distincts : 96096
customer_id unique ? True
customer_unique_id unique ? False


### Q1 — Séquence incrémentale

Sur le dataset `customers`, créer une séquence incrémentale d'entiers naturels.

In [ ]:
# row_number() sur une fenêtre triée par customer_id : séquence 1, 2, 3...
# déterministe d'une exécution à l'autre (contrairement à monotonically_increasing_id).
# Fenêtre sans partitionBy : toutes les lignes passent par une seule partition.
from pyspark.sql.window import Window
la_fenetre = Window.orderBy("customer_id")
customers = customers.withColumn("row_id", row_number().over(la_fenetre))

In [ ]:
# Aperçu de la colonne row_id
customers.show(5)

+--------------------+--------------------+------------------------+-------------+--------------+------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|row_id|
+--------------------+--------------------+------------------------+-------------+--------------+------+
|00012a2ce6f8dcda2...|248ffe10d632bebe4...|                    6273|       osasco|            SP|     1|
|000161a058600d590...|b0015e09bb4b6e47c...|                   35550|  itapecerica|            MG|     2|
|0001fd6190edaaf88...|94b11d37cd61cb299...|                   29830| nova venecia|            ES|     3|
|0002414f953443074...|4893ad4ea28b2c5b3...|                   39664|     mendonca|            MG|     4|
|000379cdec6255224...|0b83f73b19c2019e1...|                    4841|    sao paulo|            SP|     5|
+--------------------+--------------------+------------------------+-------------+--------------+------+
only showing top 5 rows


### Q2 — Filtrer les commandes livrées

Ne garder que les commandes au statut `delivered` et isoler les autres dans un dataset séparé.

In [ ]:
# Répartition des statuts de commande avant filtrage
status_counts = dfs['orders'].groupby('order_status').count()
status_counts.show()

+------------+-----+
|order_status|count|
+------------+-----+
|    invoiced|  314|
|    approved|    2|
|  processing|  301|
|   delivered|96478|
|     shipped| 1107|
| unavailable|  609|
|     created|    5|
|    canceled|  625|
+------------+-----+



In [ ]:
# Commandes livrées
orders_delivered = orders.filter(orders.order_status == 'delivered')
orders_delivered.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [ ]:
# Toutes les autres commandes (non livrées), conservées à part
orders_reste = orders.filter(orders.order_status != 'delivered')
orders_reste.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|136cce7faa42fdb2c...|ed0271e0b7da060a3...|    invoiced|     2017-04-11 12:22:08|2017-04-13 13:25:17|                        NULL|                         NULL|          2017-05-09 00:00:00|
|ee64d42b8cf066f35...|caded193e8e47b836...|     shipped|     2018-06-04 16:44:48|2018-06-05 04:31:18|         2018-06-05 14:32:00|                         NULL|          2018-06-28 00:00:00|
|0760a852e4e9d89eb...|d2a79636084590b74...|  

### Q3 — Supprimer les dates d'approbation nulles

Supprimer toutes les lignes où `order_approved_at` est nulle.

In [ ]:
# Nombre de commandes livrées sans date d'approbation
nb_nulls = orders_delivered.filter(col("order_approved_at").isNull()).count()
print("Nulls dans order_approved_at avant :", nb_nulls)

Nulls dans order_approved_at avant : 14


In [ ]:
# Suppression des lignes concernées puis vérification
orders_delivered = orders_delivered.filter(col("order_approved_at").isNotNull())
nb_nulls = orders_delivered.filter(col("order_approved_at").isNull()).count()
print("Nulls dans order_approved_at apres :", nb_nulls)

Nulls dans order_approved_at apres : 0


### Q4 — Nombre de clients uniques

Compter les clients réels distincts (`customer_unique_id`).

In [ ]:
# Nombre de clients réels distincts
nbre_distinct_cust_unique_id =customers.select('customer_unique_id').distinct().count()
print(nbre_distinct_cust_unique_id)

96096


### Q5 — Codes postaux distincts

Compter les préfixes de code postal distincts dans la table `geolocation`.

In [ ]:
# Nombre de préfixes de code postal distincts
nbre_distinct_code_geo =geolocation.select('geolocation_zip_code_prefix').distinct().count()
print(nbre_distinct_code_geo)

19015


### Q6 — Typage des dimensions produit

Convertir le poids et les dimensions des produits en `float`.

In [ ]:
# Schéma de départ : dimensions en integer
products.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)



In [ ]:
# Conversion d'une seule colonne
products = products.withColumn('product_weight_g',products.product_weight_g.cast('float'))

In [ ]:
# Généralisation : conversion de toutes les colonnes de dimensions en boucle
cols_a_convertir = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]

for c in cols_a_convertir :
  products=products.withColumn(c,col(c).cast('float'))
products.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: float (nullable = true)
 |-- product_length_cm: float (nullable = true)
 |-- product_height_cm: float (nullable = true)
 |-- product_width_cm: float (nullable = true)



### Q7 — Catégorisation par poids

Classer chaque produit dans une catégorie de poids (Léger, Moyen, Lourd, SuperLourd, Inconnu).

In [ ]:
# Classement par seuils de poids (en grammes).
# Les nulls sont traités en premier pour ne pas tomber dans otherwise().
products_catg = products.withColumn(
    'product_weight_g_categorie',
    when(col('product_weight_g').isNull(), 'Inconnu')
    .when(col('product_weight_g') < 5000, 'Leger')
    .when(col('product_weight_g') < 10000, 'Moyen')
    .when(col('product_weight_g') <= 20000, 'Lourd')
    .otherwise('SuperLourd')
)


In [ ]:
# Effectif de chaque catégorie de poids
products_catg.groupBy('product_weight_g_categorie').count().orderBy(col('count').desc()).show()

+--------------------------+-----+
|product_weight_g_categorie|count|
+--------------------------+-----+
|                     Leger|28752|
|                     Moyen| 2291|
|                     Lourd| 1421|
|                SuperLourd|  485|
|                   Inconnu|    2|
+--------------------------+-----+



### Q8 — Répartition par catégorie de produit

Compter les produits par catégorie, puis compter les mots qui composent les noms de catégories.

In [ ]:
# Nombre de produits par catégorie (610 produits sans catégorie)
products.groupBy('product_category_name').count().orderBy(col('count').desc()).show()

+---------------------+-----+
|product_category_name|count|
+---------------------+-----+
|      cama_mesa_banho| 3029|
|        esporte_lazer| 2867|
|     moveis_decoracao| 2657|
|         beleza_saude| 2444|
| utilidades_domest...| 2335|
|           automotivo| 1900|
| informatica_acess...| 1639|
|           brinquedos| 1411|
|   relogios_presentes| 1329|
|            telefonia| 1134|
|                bebes|  919|
|           perfumaria|  868|
|            papelaria|  849|
| fashion_bolsas_e_...|  849|
|           cool_stuff|  789|
|   ferramentas_jardim|  753|
|             pet_shop|  719|
|                 NULL|  610|
|          eletronicos|  517|
| construcao_ferram...|  400|
+---------------------+-----+
only showing top 20 rows


In [ ]:
# Découpage des noms de catégorie sur "_" puis comptage de chaque mot
products.select(explode(split('product_category_name','_')).alias("mot")).groupBy("mot").count().show(5)

+----------+-----+
|       mot|count|
+----------+-----+
|      cama| 3029|
|escritorio|  309|
| papelaria|  849|
|    bolsas|  849|
|  relogios| 1329|
+----------+-----+
only showing top 5 rows


### Q9 — Volume et rang de poids

Calculer le volume de chaque produit (L × H × l) et son rang selon le poids.

In [ ]:
# Première version (gardée pour référence) : le dense_rank et le tri par volume
# étaient appliqués directement sur products. Remplacée par la cellule suivante.
expr = col('product_length_cm') *col('product_height_cm') * col('product_width_cm')
fenetre_2 = Window.orderBy(col('product_weight_g').desc())
#products = products.withColumn('Volume',expr).orderBy(col('Volume').desc())
#products = products.withColumn('Rang_Poids',dense_rank().over(fenetre_2))

#products.select('product_id','product_weight_g','Volume','Rang_Poids').show(10)

In [ ]:
# Volume = longueur x hauteur x largeur (cm3).
# rank() sur le poids décroissant : les ex aequo partagent le même rang
# et le rang suivant saute (ex. : tous les produits de 30 000 g ont le rang 2).
expr = col('product_length_cm') * col('product_height_cm') * col('product_width_cm')
fenetre_2 = Window.orderBy(col('product_weight_g').desc())

products_vol = (
    products
    .withColumn('Volume', expr)
    .withColumn('Rang_Poids', rank().over(fenetre_2))
)

products_vol.orderBy(col('Volume').desc()) \
    .select('product_id', 'product_weight_g', 'Volume', 'Rang_Poids') \
    .show(10)

+--------------------+----------------+--------+----------+
|          product_id|product_weight_g|  Volume|Rang_Poids|
+--------------------+----------------+--------+----------+
|256a9c364b75753b9...|         25250.0|296208.0|       237|
|f227e2d44f10f7dad...|         30000.0|294000.0|         2|
|3eb14e65e4208c6d9...|         30000.0|294000.0|         2|
|c1e0531cb1864fd3a...|         30000.0|294000.0|         2|
|0b48eade13cfad433...|         14000.0|294000.0|      1107|
|90c1b4e040d1d1c45...|         30000.0|293706.0|         2|
|99ff40856c47a638d...|         30000.0|288000.0|         2|
|8d6f2c3454002d3f5...|         30000.0|288000.0|         2|
|c6fdec160d0f8f488...|         30000.0|288000.0|         2|
|0e9dfb804bafa3d68...|         30000.0|287980.0|         2|
+--------------------+----------------+--------+----------+
only showing top 10 rows


### Q10 — Contrôle des prix

Vérifier l'absence de prix négatifs dans `order_items`.

In [ ]:
# Aperçu de la table des articles
order_items.show(4)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30|199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18|12.99|        12.79|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
only showing top 4 rows


In [ ]:
# Contrôle qualité : nombre de prix négatifs
price_negatif = order_items.filter(order_items.price < 0).count()
print(price_negatif)

0


---
<a id="exercice-2"></a>
## Partie 2 — Niveau intermédiaire : jointures et agrégations

Objectif : construire une **table de faits** pour analyser la performance des vendeurs.

### Q1 — Jointure `order_items` × `orders` × `sellers`

In [ ]:
# Jointures en left pour ne perdre aucune commande :
# 1) articles + vendeurs, 2) commandes + (articles + vendeurs).
# Une commande avec plusieurs articles donne plusieurs lignes.
items_sellers = order_items.join(sellers, on="seller_id", how="left")
orders_items_sellers = orders.join(items_sellers, on="order_id", how="left")
orders_items_sellers.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+-------------+--------------------+-------------------+-----+-------------+----------------------+--------------------+------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|           seller_id|order_item_id|          product_id|shipping_limit_date|price|freight_value|seller_zip_code_prefix|         seller_city|seller_state|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+-------------+--------------------+-------------------+-----+-------------+----------------------+---

In [ ]:
# Mise en cache en mémoire : cette table est réutilisée dans plusieurs questions.
# persist() est paresseux, le count() force le calcul et le stockage.
from pyspark import StorageLevel
orders_items_sellers.persist(StorageLevel.MEMORY_ONLY)
print(orders_items_sellers.count())  # déclenche le stockage

113425


In [ ]:
# Variante : une ligne par commande, les informations articles/vendeurs
# regroupées dans des tableaux (collect_list).
# Contrôle : on retrouve bien une ligne par commande.
cols_orders = orders_delivered.columns
cols_items = [c for c in items_sellers.columns if c != "order_id"]

aggs = [collect_list(c).alias(c) for c in cols_items]

orders_arrays = orders_items_sellers.groupBy(*cols_orders).agg(*aggs)

# Contrôles
print("orders_delivered :", orders.count())
print("orders_arrays :", orders_arrays.count())
orders_arrays.show()

orders_delivered : 99441
orders_arrays : 99441
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+-------------+--------------------+--------------------+--------------------+--------------------+----------------------+--------------------+----------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|           seller_id|order_item_id|          product_id| shipping_limit_date|               price|       freight_value|seller_zip_code_prefix|         seller_city|    seller_state|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+

### Q2 — Chiffre d'affaires et nombre de commandes par vendeur

In [ ]:
from pyspark.sql.functions import countDistinct
# 1. Répartition des statuts : voir ce que contient la table avant de décider du périmètre
orders_items_sellers.groupBy("order_status").count().orderBy(col("count").desc()).show()

# 2. Périmètre du CA : commandes livrées uniquement, et lignes avec un vendeur connu
#    (le left join depuis orders laisse seller_id null pour les commandes sans article)
base_ca = orders_items_sellers.filter(
    (col("order_status") == "delivered") & col("seller_id").isNotNull()
)

# 3. Agrégation par vendeur
#    - CA = somme des prix (frais de port exclus)
#    - nb_commandes = commandes DISTINCTES, car un vendeur peut avoir
#      plusieurs articles dans la même commande
ca_vendeurs = (
    base_ca
    .groupBy("seller_id")
    .agg(
        sum("price").alias("ca_total"),
        countDistinct("order_id").alias("nb_commandes"),
    )
    .orderBy(col("ca_total").desc())
)

ca_vendeurs.show(10)

# 4. Contrôles
print("Vendeurs dans le résultat :", ca_vendeurs.count())
print("Vendeurs distincts dans base_ca :", base_ca.select("seller_id").distinct().count())
print("Somme des ca_total :", ca_vendeurs.agg(sum("ca_total")).first()[0])
print("Somme de price :", base_ca.agg(sum("price")).first()[0])

+------------+------+
|order_status| count|
+------------+------+
|   delivered|110197|
|     shipped|  1186|
|    canceled|   706|
| unavailable|   610|
|    invoiced|   361|
|  processing|   357|
|     created|     5|
|    approved|     3|
+------------+------+

+--------------------+------------------+------------+
|           seller_id|          ca_total|nb_commandes|
+--------------------+------------------+------------+
|4869f7a5dfa277a7d...|226987.93000000023|        1124|
|53243585a1d6dc264...|217940.44000000006|         348|
|4a3ca9315b744ce9f...|196882.12000000032|        1772|
|fa1c13f2614d7b5c4...|190917.14000000007|         578|
|7c67e1448b00f6e96...| 186570.0500000001|         973|
|7e93a43ef30c4f03f...|165981.49000000002|         319|
|da8622b14eb17ae28...|159816.87000000005|        1311|
|7a67c85e85bb2ce85...| 139658.6899999999|        1145|
|1025f0e2d44d7041d...|138208.55999999997|         910|
|955fee9216a65b617...|131836.70999999993|        1261|
+-------------------

### Q3 — Panier moyen par vendeur

Panier moyen = CA / nombre de commandes.

In [ ]:
# round() sans précision : panier moyen arrondi à l'unité
expr = round(ca_vendeurs.ca_total / ca_vendeurs.nb_commandes)
ca_vendeurs = ca_vendeurs.withColumn('panier_moyen',expr)
ca_vendeurs.show()

+--------------------+------------------+------------+------------+
|           seller_id|          ca_total|nb_commandes|panier_moyen|
+--------------------+------------------+------------+------------+
|4869f7a5dfa277a7d...|226987.93000000023|        1124|       202.0|
|53243585a1d6dc264...|217940.44000000006|         348|       626.0|
|4a3ca9315b744ce9f...|196882.12000000032|        1772|       111.0|
|fa1c13f2614d7b5c4...|190917.14000000007|         578|       330.0|
|7c67e1448b00f6e96...| 186570.0500000001|         973|       192.0|
|7e93a43ef30c4f03f...|165981.49000000002|         319|       520.0|
|da8622b14eb17ae28...|159816.87000000005|        1311|       122.0|
|7a67c85e85bb2ce85...| 139658.6899999999|        1145|       122.0|
|1025f0e2d44d7041d...|138208.55999999997|         910|       152.0|
|955fee9216a65b617...|131836.70999999993|        1261|       105.0|
|46dc3b2cc0980fb8e...|122811.37999999995|         503|       244.0|
|6560211a19b47992c...|         120702.83|       

### Q4 — Tri par CA et sauvegarde Parquet

In [ ]:
# Écriture Parquet (overwrite pour pouvoir relancer la cellule)
ca_vendeurs_trie = ca_vendeurs.orderBy(col("ca_total").desc())

ca_vendeurs_trie.write.mode("overwrite").parquet("output/ca_vendeurs")

In [ ]:
# Relecture du Parquet pour vérifier le contenu écrit
verif = spark.read.parquet("output/ca_vendeurs")
print(verif.count())
verif.show(5)

2970
+--------------------+------------------+------------+------------+
|           seller_id|          ca_total|nb_commandes|panier_moyen|
+--------------------+------------------+------------+------------+
|4869f7a5dfa277a7d...|226987.93000000023|        1124|       202.0|
|53243585a1d6dc264...|217940.44000000006|         348|       626.0|
|4a3ca9315b744ce9f...|196882.12000000032|        1772|       111.0|
|fa1c13f2614d7b5c4...|190917.14000000007|         578|       330.0|
|7c67e1448b00f6e96...| 186570.0500000001|         973|       192.0|
+--------------------+------------------+------------+------------+
only showing top 5 rows


### Q5 — Délai réel de livraison

`order_delivered_customer_date - order_purchase_timestamp`, pour chaque commande.

In [ ]:
# Rappel des colonnes de dates disponibles
orders_items_sellers.show(3)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+-------------+--------------------+-------------------+-----+-------------+----------------------+-----------+------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|           seller_id|order_item_id|          product_id|shipping_limit_date|price|freight_value|seller_zip_code_prefix|seller_city|seller_state|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+-------------+--------------------+-------------------+-----+-------------+----------------------+-----------+---------

In [ ]:
# datediff() renvoie un nombre de jours entiers.
# Null pour les commandes non livrées (pas de date de livraison).
orders_arrays = orders_arrays.withColumn('temps_livraison', datediff(col('order_delivered_customer_date'),col('order_purchase_timestamp')))
orders_arrays.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+-------------+--------------------+--------------------+--------------------+--------------------+----------------------+--------------------+----------------+---------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|           seller_id|order_item_id|          product_id| shipping_limit_date|               price|       freight_value|seller_zip_code_prefix|         seller_city|    seller_state|temps_livraison|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+-------------+-

### Q6 — Indicateur de retard `is_late`

Booléen indiquant si la livraison a dépassé la date estimée.

In [ ]:
# True si la livraison réelle dépasse la date estimée.
# Reste null quand la commande n'a pas de date de livraison.
orders_arrays = orders_arrays.withColumn('is_late', col('order_delivered_customer_date')>col('order_estimated_delivery_date'))

In [ ]:
# Répartition : en retard / à l'heure / non livré
orders_arrays.groupBy("is_late").count().show()

+-------+-----+
|is_late|count|
+-------+-----+
|   NULL| 2965|
|   true| 7827|
|  false|88649|
+-------+-----+



### Q7 — Jointure optimisée `orders` × `order_reviews`

Comparer une jointure classique (Sort Merge Join) et une jointure broadcast.

In [ ]:
# Comparaison des deux stratégies de jointure.
# 1) Broadcast automatique désactivé (seuil -1) pour forcer un SortMergeJoin.
# 2) broadcast() explicite : la petite table est envoyée à chaque exécuteur,
#    ce qui évite le shuffle (BroadcastHashJoin dans le plan).
# Sur ce volume et en local, le gain de temps n'est pas mesurable ; l'intérêt
# se voit dans le plan d'exécution (plus d'Exchange ni de Sort).
# Le seuil par défaut (10 Mo) est rétabli à la fin.
import time
# join classique
spark.conf.set('spark.sql.autoBroadcastJoinThreshold',-1)
t0=time.time()
classique=orders.join(order_reviews, on="order_id", how="left")
print("lignes :", classique.count())
print(f"temps classique : {time.time() - t0:.2f} s")
classique.explain()
# join optimise
t0 =time.time()
optimisee = orders.join(broadcast(order_reviews), on='order_id',how='left')

print("lignes :", optimisee.count())
print(f"temps broadcast : {time.time() - t0:.2f} s")
optimisee.explain()   # voir BroadcastHashJoin

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10485760)


lignes : 99992
temps classique : 1.99 s
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [order_id#180, customer_id#181, order_status#182, order_purchase_timestamp#183, order_approved_at#184, order_delivered_carrier_date#185, order_delivered_customer_date#186, order_estimated_delivery_date#187, review_id#145, review_score#147, review_comment_title#148, review_comment_message#149, review_creation_date#150, review_answer_timestamp#151]
   +- SortMergeJoin [order_id#180], [order_id#146], LeftOuter
      :- Sort [order_id#180 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(order_id#180, 8), ENSURE_REQUIREMENTS, [plan_id=3340]
      :     +- FileScan csv [order_id#180,customer_id#181,order_status#182,order_purchase_timestamp#183,order_approved_at#184,order_delivered_carrier_date#185,order_delivered_customer_date#186,order_estimated_delivery_date#187] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/olist_data

### Q8 — Nombre de commandes par mois et par année

In [ ]:
# Extraction de l'année et du mois d'achat puis comptage des commandes
commandes_months_year = orders.select(
		'order_id',
		extract(lit('YEAR'), 'order_purchase_timestamp').alias('year'),
		extract(lit('MONTH'), 'order_purchase_timestamp').alias('month')

).groupby('year','month').agg(count('order_id').alias('nbre_commandes')).orderBy(col('year'),col('month'))

In [ ]:
# Aperçu chronologique
commandes_months_year.show()

+----+-----+--------------+
|year|month|nbre_commandes|
+----+-----+--------------+
|2016|    9|             4|
|2016|   10|           324|
|2016|   12|             1|
|2017|    1|           800|
|2017|    2|          1780|
|2017|    3|          2682|
|2017|    4|          2404|
|2017|    5|          3700|
|2017|    6|          3245|
|2017|    7|          4026|
|2017|    8|          4331|
|2017|    9|          4285|
|2017|   10|          4631|
|2017|   11|          7544|
|2017|   12|          5673|
|2018|    1|          7269|
|2018|    2|          6728|
|2018|    3|          7211|
|2018|    4|          6939|
|2018|    5|          6873|
+----+-----+--------------+
only showing top 20 rows


In [ ]:
# 1. La somme des comptages doit être égale au nombre de commandes
total_tableau = commandes_months_year.agg(sum('nbre_commandes')).first()[0]
total_orders = orders.count()
print("Somme du tableau :", total_tableau)
print("orders.count()   :", total_orders)
print("OK ?", total_tableau == total_orders)

# 2. Aucune date d'achat nulle (sinon des commandes tomberaient dans un groupe year/month = null)
print("Dates d'achat nulles :", orders.filter(col('order_purchase_timestamp').isNull()).count())

# 3. Nombre de mois dans le tableau et lecture chronologique
print("Nombre de mois :", commandes_months_year.count())
commandes_months_year.show(30)

Somme du tableau : 99441
orders.count()   : 99441
OK ? True
Dates d'achat nulles : 0
Nombre de mois : 25
+----+-----+--------------+
|year|month|nbre_commandes|
+----+-----+--------------+
|2016|    9|             4|
|2016|   10|           324|
|2016|   12|             1|
|2017|    1|           800|
|2017|    2|          1780|
|2017|    3|          2682|
|2017|    4|          2404|
|2017|    5|          3700|
|2017|    6|          3245|
|2017|    7|          4026|
|2017|    8|          4331|
|2017|    9|          4285|
|2017|   10|          4631|
|2017|   11|          7544|
|2017|   12|          5673|
|2018|    1|          7269|
|2018|    2|          6728|
|2018|    3|          7211|
|2018|    4|          6939|
|2018|    5|          6873|
|2018|    6|          6167|
|2018|    7|          6292|
|2018|    8|          6512|
|2018|    9|            16|
|2018|   10|             4|
+----+-----+--------------+



### Q9 — Top 5 des villes vendeuses aux frais de port moyens les plus élevés

In [ ]:
# Première approche : moyenne brute par ville
top_5_villes = orders_items_sellers.groupBy("seller_city").agg(avg('freight_value').alias('frais_port_moyen')).orderBy(col('frais_port_moyen').desc()).limit(5)
top_5_villes.show()

+--------------------+------------------+
|         seller_city|  frais_port_moyen|
+--------------------+------------------+
|          lages - sc|168.53333333333333|
|sao francisco do sul|            150.22|
|          california|           143.775|
|sao  jose dos pin...|142.39999999999998|
|         nova trento|            131.85|
+--------------------+------------------+



In [ ]:
# Version corrigée : les villes avec très peu d'articles faussent le classement
# (une moyenne sur 2 ou 3 articles n'est pas représentative).
# On normalise le nom des villes et on impose un effectif minimum.
# Seuil minimum d'articles pour qu'une moyenne soit jugée fiable (à ajuster)
SEUIL = 30

villes = (
    orders_items_sellers
    .filter(col("seller_city").isNotNull())                       # écarte les commandes sans article
    .withColumn("seller_city", lower(trim(col("seller_city"))))   # unifie majuscules et espaces
)

stats_villes = (
    villes
    .groupBy("seller_city")
    .agg(
        avg("freight_value").alias("frais_port_moyen"),
        count("order_id").alias("nb_articles"),
    )
)

# Top 5 brut (sans seuil), pour voir le problème des petits effectifs
print("Top 5 sans seuil :")
stats_villes.orderBy(col("frais_port_moyen").desc()).show(5, truncate=False)

# Top 5 fiable (avec seuil)
top_5_villes = (
    stats_villes
    .filter(col("nb_articles") >= SEUIL)
    .orderBy(col("frais_port_moyen").desc())
    .limit(5)
)
print(f"Top 5 avec au moins {SEUIL} articles :")
top_5_villes.show(truncate=False)

# Repère : la moyenne générale
print("Moyenne générale des frais de port :",
      villes.agg(avg("freight_value")).first()[0])

Top 5 sans seuil :
+---------------------+------------------+-----------+
|seller_city          |frais_port_moyen  |nb_articles|
+---------------------+------------------+-----------+
|lages - sc           |168.53333333333333|3          |
|sao francisco do sul |150.22            |2          |
|california           |143.775           |6          |
|sao  jose dos pinhais|142.39999999999998|2          |
|nova trento          |131.85            |2          |
+---------------------+------------------+-----------+
only showing top 5 rows
Top 5 avec au moins 30 articles :
+------------+-----------------+-----------+
|seller_city |frais_port_moyen |nb_articles|
+------------+-----------------+-----------+
|fortaleza   |60.3388          |50         |
|cachoeirinha|59.61522388059702|67         |
|vicosa      |52.33431818181818|44         |
|lages       |49.39229508196722|122        |
|mogi guacu  |47.40307359307359|231        |
+------------+-----------------+-----------+

Moyenne générale des f

### Q10 — Valeurs manquantes dans les avis

Remplacer les commentaires manquants par `"Pas de commentaire"`.

In [ ]:
# na.fill limité à la colonne review_comment_message.
# Remarque : la ligne dont review_id commence par "," vient de commentaires
# multi-lignes mal découpés à la lecture du CSV.
order_reviews = order_reviews.na.fill("Pas de commentaire",'review_comment_message')
order_reviews.show()


+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|           review_id|            order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|7bc2406110b926393...|73fc7af87114b3971...|           4|                NULL|    Pas de commentaire| 2018-01-18 00:00:00|    2018-01-18 21:46:59|
|80e641a11e56f04c1...|a548910a1c6147796...|           5|                NULL|    Pas de commentaire| 2018-03-10 00:00:00|    2018-03-11 03:05:13|
|228ce5500dc1d8e02...|f9e4b658b201a9f2e...|           5|                NULL|    Pas de commentaire| 2018-02-17 00:00:00|    2018-02-18 14:36:24|
|e64fb393e7b32834b...|658677c97b385a9be...|           5|                NULL|  Recebi bem antes ...| 2017-04-21 00:00:00|   

---
<a id="exercice-3"></a>
## Partie 3 — Niveau avancé : analyse client et optimisation

Objectif : aller plus loin avec les fonctions de fenêtre, le pivot, le partitionnement
à l'écriture et la détection de valeurs extrêmes.

### Q1 — Délai entre la 1re et la 2e commande

Pour chaque client (`customer_unique_id`), calculer l'intervalle en jours entre sa première et sa deuxième commande.

In [ ]:
# Fenêtre par client réel, triée par date d'achat (order_id pour départager les ex aequo).
# row_number() numérote les commandes, lag() récupère la date de la commande précédente.
# Sur la 2e commande, l'écart entre les deux dates donne l'intervalle recherché.
orders_customers = orders.join(customers, on ="customer_id", how="left")
fenetre_3 = Window.partitionBy("customer_unique_id").orderBy("order_purchase_timestamp", "order_id")
orders_customers = orders_customers.withColumns({'rang_cmd':row_number().over(fenetre_3)
						,'date_cmd_1':lag("order_purchase_timestamp", offset=1, default=None).over(fenetre_3)})

intervalle_cmd = orders_customers.filter('rang_cmd==2').withColumn('intervalle_temps_cmd',datediff("order_purchase_timestamp",'date_cmd_1'))
intervalle_cmd.select('customer_unique_id','order_purchase_timestamp','date_cmd_1','rang_cmd','intervalle_temps_cmd').show(40)

+--------------------+------------------------+-------------------+--------+--------------------+
|  customer_unique_id|order_purchase_timestamp|         date_cmd_1|rang_cmd|intervalle_temps_cmd|
+--------------------+------------------------+-------------------+--------+--------------------+
|00172711b30d52eea...|     2018-08-13 09:14:07|2018-07-28 00:23:49|       2|                  16|
|004288347e5e88a27...|     2018-01-14 07:36:54|2017-07-27 14:13:03|       2|                 171|
|004b45ec5c6418746...|     2018-05-26 19:42:48|2017-09-01 12:11:23|       2|                 267|
|0058f300f57d7b93c...|     2018-03-22 18:09:41|2018-02-19 17:11:34|       2|                  31|
|00a39521eb40f7012...|     2018-06-03 10:12:57|2018-05-23 20:14:21|       2|                  11|
|012a218df8995d3ec...|     2018-06-18 13:08:38|2018-05-07 10:28:17|       2|                  42|
|013f4353d26bb05dc...|     2017-11-28 13:30:58|2017-11-24 13:33:20|       2|                   4|
|01886ef98f995e4f2..

In [ ]:
# Contrôles de cohérence du résultat
# 1. Aucun écart négatif, et ordre de grandeur des écarts
intervalle_cmd.agg(
    min('intervalle_temps_cmd').alias('min_jours'),
    max('intervalle_temps_cmd').alias('max_jours'),
    avg('intervalle_temps_cmd').alias('moyenne_jours'),
).show()

# 2. Taux de réachat : clients ayant au moins 2 commandes / tous les clients
nb_clients = customers.select('customer_unique_id').distinct().count()
nb_reachat = intervalle_cmd.count()
print(f"Clients : {nb_clients} | avec 2e commande : {nb_reachat} | taux : {nb_reachat / nb_clients:.2%}")

# 3. Un seul rang 2 par client (sinon la fenêtre est mal définie)
print("Doublons de client :", intervalle_cmd.count() - intervalle_cmd.select('customer_unique_id').distinct().count())

# 4. Contrôle manuel : les commandes d'un client concerné, dans l'ordre
un_client = intervalle_cmd.select('customer_unique_id').first()[0]
orders_customers.filter(col('customer_unique_id') == un_client) \
    .select('customer_unique_id', 'order_id', 'order_purchase_timestamp', 'rang_cmd', 'date_cmd_1') \
    .orderBy('rang_cmd').show(truncate=False)

+---------+---------+-----------------+
|min_jours|max_jours|    moyenne_jours|
+---------+---------+-----------------+
|        0|      609|80.34834834834835|
+---------+---------+-----------------+

Clients : 96096 | avec 2e commande : 2997 | taux : 3.12%
Doublons de client : 0
+--------------------------------+--------------------------------+------------------------+--------+-------------------+
|customer_unique_id              |order_id                        |order_purchase_timestamp|rang_cmd|date_cmd_1         |
+--------------------------------+--------------------------------+------------------------+--------+-------------------+
|00172711b30d52eea8b313a7f2cced02|bb874c45df1a3c97842d52f31efee99a|2018-07-28 00:23:49     |1       |NULL               |
|00172711b30d52eea8b313a7f2cced02|c306eca42d32507b970739b5b6a5a33a|2018-08-13 09:14:07     |2       |2018-07-28 00:23:49|
+--------------------------------+--------------------------------+------------------------+--------+--------

### Q2 — Clients représentant 80 % du chiffre d'affaires (Pareto)

In [ ]:
# Schémas des tables utilisées
order_items.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)



In [ ]:
customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- row_id: integer (nullable = false)



In [ ]:
# Analyse de Pareto :
# 1) CA par client réel (commandes livrées uniquement)
# 2) cumul du CA en triant les clients du plus gros au plus petit
# 3) on garde un client tant que le CA cumulé AVANT lui est < 80 %,
#    ce qui inclut le client qui fait franchir le seuil.
orders_items_customers = order_items.join(orders , on ='order_id', how='left').join(customers,on='customer_id', how='left')

ca_clients = orders_items_customers.filter('order_status == "delivered"').groupBy('customer_unique_id').agg(sum(col('price')).alias('CA_client'))

CA_total = ca_clients.agg(sum("CA_client")).first()[0]
fenetre_cumul = Window.orderBy(col("CA_client").desc(), "customer_unique_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)

ca_clients_cumul = ca_clients.withColumn("CA_cumul", sum("CA_client").over(fenetre_cumul))

ca_clients_cumul = ca_clients_cumul.withColumns({'pct_cumul':col('CA_cumul') / CA_total ,
'pct_avant':(col("CA_cumul") - col("CA_client")) / CA_total})

clients_80 = ca_clients_cumul.filter('pct_avant< 0.8').orderBy(col("CA_client").desc())
clients_80.show()

+--------------------+---------+---------+--------------------+--------------------+
|  customer_unique_id|CA_client| CA_cumul|           pct_cumul|           pct_avant|
+--------------------+---------+---------+--------------------+--------------------+
|0a0a92112bd4c708c...|  13440.0|  13440.0|0.001016526258081...|                 0.0|
|da122df9eeddfedc1...|   7388.0|  20828.0|0.001575313162450...|0.001016526258081...|
|763c8b1c9c68a0229...|   7160.0|  27988.0|0.002116855424940096|0.001575313162450...|
|dc4802a71eae9be1d...|   6735.0|  34723.0| 0.00262625306989406|0.002116855424940096|
|459bef486812aa252...|   6729.0|  41452.0|0.003135196908482...| 0.00262625306989406|
|ff4159b92c40ebe40...|   6499.0|  47951.0|0.003626744836404...|0.003135196908482...|
|4007669dec559734d...|   5934.6|  53885.6|0.004075604712239247|0.003626744836404...|
|eebb5dda148d3893c...|   4690.0|  58575.6| 0.00443033002104906|0.004075604712239247|
|48e1ac109decbb877...|   4590.0|  63165.6|0.004777491890438...| 0

In [ ]:
# Contrôles : part des clients retenus, couverture du CA et frontière des 80 %
# Rappel : clients_80 doit être défini SANS .show() à la fin
# clients_80 = ca_clients_cumul.filter(col("pct_avant") < 0.8)

nb_total = ca_clients.count()
nb_80 = clients_80.count()

# 1. Nombre et part de clients retenus
print(f"Clients retenus : {nb_80} sur {nb_total} ({nb_80 / nb_total:.2%})")

# 2. Le CA des clients retenus doit être un peu au-dessus de 80 %
ca_80 = clients_80.agg(sum("CA_client")).first()[0]
print(f"Part du CA couverte : {ca_80 / CA_total:.4%}")

# 3. Le dernier client gardé : pct_avant juste sous 0.8, pct_cumul juste au-dessus
clients_80.orderBy(col("pct_cumul").desc()).select(
    "customer_unique_id", "CA_client", "pct_avant", "pct_cumul"
).show(3)

# 4. Le premier client exclu : son pct_avant doit être >= 0.8
ca_clients_cumul.filter(col("pct_avant") >= 0.8).orderBy("pct_avant").select(
    "customer_unique_id", "CA_client", "pct_avant", "pct_cumul"
).show(3)

# 5. Le cumul total doit être égal au CA total, et pct_cumul finir à 1
print("Max CA_cumul :", ca_clients_cumul.agg(max("CA_cumul")).first()[0], "| CA_total :", CA_total)
print("Max pct_cumul :", ca_clients_cumul.agg(max("pct_cumul")).first()[0])

Clients retenus : 42009 sur 93358 (45.00%)
Part du CA couverte : 80.0002%
+--------------------+---------+------------------+------------------+
|  customer_unique_id|CA_client|         pct_avant|         pct_cumul|
+--------------------+---------+------------------+------------------+
|0255fc149c1f19131...|     99.0|0.7999946505306897|0.8000021383357158|
|022f73728cb75b707...|     99.0|0.7999871627256636|0.7999946505306897|
|01f382f4c95df31e0...|     99.0|0.7999796749206376|0.7999871627256636|
+--------------------+---------+------------------+------------------+
only showing top 3 rows
+--------------------+---------+------------------+------------------+
|  customer_unique_id|CA_client|         pct_avant|         pct_cumul|
+--------------------+---------+------------------+------------------+
|02c8bc3ccd6019dbd...|     99.0|0.8000021383357158|0.8000096261407418|
|0301e2b22f44b5a06...|     99.0|0.8000096261407418|0.8000171139457679|
|0376362c3d6ae393d...|     99.0|0.8000171139457679

### Q3 — Tableau croisé : montant payé par type de paiement et par mois

In [ ]:
# Schéma de la table des paiements
order_payments.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)



In [ ]:
# Première version du pivot : mois en lignes, type de paiement en colonnes.
# La commande sans paiement fait apparaître une colonne "null".
orders_pay_join = orders.join(order_payments, on ='order_id',how='left')
orders_pay_mois=orders_pay_join.withColumn('mois',date_format(col('order_purchase_timestamp'),"yyyy-MM"))
montants_mois=orders_pay_mois.groupBy('mois').pivot('payment_type').agg(round(sum('payment_value'), 2).alias('montant_total')).orderBy(col('mois'))
montants_mois.show()

+-------+----+---------+-----------+----------+-----------+--------+
|   mois|null|   boleto|credit_card|debit_card|not_defined| voucher|
+-------+----+---------+-----------+----------+-----------+--------+
|2016-09|NULL|     NULL|     252.24|      NULL|       NULL|    NULL|
|2016-10|NULL|  9679.06|   48290.62|    241.73|       NULL|  879.07|
|2016-12|NULL|     NULL|      19.62|      NULL|       NULL|    NULL|
|2017-01|NULL| 24074.43|  109615.68|    743.53|       NULL|  4054.4|
|2017-02|NULL| 57476.74|  226753.56|   1510.32|       NULL| 6167.39|
|2017-03|NULL| 82160.53|  354488.94|    3592.8|       NULL| 9621.33|
|2017-04|NULL| 81307.24|  322087.72|    2790.0|       NULL|11603.07|
|2017-05|NULL| 128652.9|  445047.31|   3023.41|       NULL| 16195.2|
|2017-06|NULL|107774.16|  388268.27|   2424.17|       NULL|12809.78|
|2017-07|NULL|116775.48|  454988.47|   2345.03|       NULL|18273.94|
|2017-08|NULL|134289.81|  521595.69|   3136.84|       NULL|15373.98|
|2017-09|NULL|131042.24|  574484.1

In [ ]:
# Version finale : sans la colonne null et cases vides remplacées par 0.
# (reduce et add sont importés mais finalement non utilisés)
from functools import reduce
from operator import add

# Combien de commandes n'ont aucun paiement ? (elles créent la colonne "null")
print("Lignes sans payment_type :", orders_pay_mois.filter(col("payment_type").isNull()).count())

# Tableau croisé : une ligne par mois, une colonne par type de paiement
tableau_croise = (
    orders_pay_mois
    .filter(col("payment_type").isNotNull())            # supprime la colonne "null"
    .groupBy("mois")
    .pivot("payment_type")
    .agg(round(sum("payment_value"), 2))                # montants arrondis à 2 décimales
    .na.fill(0)                                         # les cases vides deviennent 0
    .orderBy("mois")
)

tableau_croise.show(30)

Lignes sans payment_type : 1
+-------+---------+-----------+----------+-----------+--------+
|   mois|   boleto|credit_card|debit_card|not_defined| voucher|
+-------+---------+-----------+----------+-----------+--------+
|2016-09|      0.0|     252.24|       0.0|        0.0|     0.0|
|2016-10|  9679.06|   48290.62|    241.73|        0.0|  879.07|
|2016-12|      0.0|      19.62|       0.0|        0.0|     0.0|
|2017-01| 24074.43|  109615.68|    743.53|        0.0|  4054.4|
|2017-02| 57476.74|  226753.56|   1510.32|        0.0| 6167.39|
|2017-03| 82160.53|  354488.94|    3592.8|        0.0| 9621.33|
|2017-04| 81307.24|  322087.72|    2790.0|        0.0|11603.07|
|2017-05| 128652.9|  445047.31|   3023.41|        0.0| 16195.2|
|2017-06|107774.16|  388268.27|   2424.17|        0.0|12809.78|
|2017-07|116775.48|  454988.47|   2345.03|        0.0|18273.94|
|2017-08|134289.81|  521595.69|   3136.84|        0.0|15373.98|
|2017-09|131042.24|  574484.15|   4690.36|        0.0| 17545.7|
|2017-10|14

### Q4 — Évolution mensuelle du volume de commandes (montant et nombre)

In [ ]:
# Rappel de la structure de la table commandes + paiements
orders_pay_mois.show(1)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+------------------+------------+--------------------+-------------+-------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|payment_sequential|payment_type|payment_installments|payment_value|   mois|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+------------------+------------+--------------------+-------------+-------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|            

In [ ]:
# Montant total payé et nombre de commandes distinctes par mois.
# countDistinct est nécessaire : une commande peut avoir plusieurs paiements.
volume = orders_pay_mois.groupBy('mois').agg(sum('payment_value').alias('montant_total'),countDistinct('order_id').alias('nbre_cmd')).orderBy('mois')
volume.show()

+-------+------------------+--------+
|   mois|     montant_total|nbre_cmd|
+-------+------------------+--------+
|2016-09|            252.24|       4|
|2016-10| 59090.47999999999|     324|
|2016-12|             19.62|       1|
|2017-01|138488.03999999998|     800|
|2017-02|291908.00999999995|    1780|
|2017-03|449863.60000000003|    2682|
|2017-04| 417788.0300000003|    2404|
|2017-05| 592918.8200000009|    3700|
|2017-06| 511276.3800000001|    3245|
|2017-07| 592382.9200000016|    4026|
|2017-08| 674396.3200000026|    4331|
|2017-09| 727762.4500000023|    4285|
|2017-10| 779677.8800000027|    4631|
|2017-11|1194882.8000000017|    7544|
|2017-12| 878401.4800000032|    5673|
|2018-01|1115004.1800000048|    7269|
|2018-02| 992463.3400000032|    6728|
|2018-03|1159652.1199999978|    7211|
|2018-04|1160785.4800000018|    6939|
|2018-05|1153982.1500000008|    6873|
+-------+------------------+--------+
only showing top 20 rows


### Q5 — Écriture de la table de faits partitionnée par `customer_state`

In [ ]:
# Table de faits = commandes x articles x vendeurs x clients.
# partitionBy crée un sous-dossier par État (customer_state=SP, ...) :
# une requête filtrée sur un État ne lit que son dossier (partition pruning).
table_fait = orders_items_sellers.join(customers,on='customer_id',how='left')
table_fait.write.mode('overwrite').partitionBy("customer_state").parquet('output/table_fait')

In [ ]:
# Aperçu de la table de faits
table_fait.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+-------------+--------------------+-------------------+------+-------------+----------------------+---------------+------------+--------------------+------------------------+--------------------+--------------+------+
|         customer_id|            order_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|           seller_id|order_item_id|          product_id|shipping_limit_date| price|freight_value|seller_zip_code_prefix|    seller_city|seller_state|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|row_id|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+------------

### Q6 — Commandes extrêmes

Commandes dont le montant dépasse la moyenne de plus de 3 écarts-types.

In [ ]:
# 1) Montant de chaque commande = somme des prix de ses articles
# 2) Seuil = moyenne + 3 x écart-type
# 3) Colonne booléenne indiquant si la commande dépasse ce seuil
table_montant_cmd = table_fait.filter(col("price").isNotNull()).groupBy('order_id').agg(sum("price").alias("montant_commande"))

moyenne_montant = table_montant_cmd.agg(avg("montant_commande").alias('moyenne')).first()[0]

ecartType_montant = table_montant_cmd.agg(stddev("montant_commande").alias('ecartType')).first()[0]

expr = moyenne_montant + (3*ecartType_montant)

table_mt_ecartType = table_montant_cmd.withColumn('calcul_verif',col('montant_commande')> expr)
table_mt_ecartType.show()

+--------------------+----------------+------------+
|            order_id|montant_commande|calcul_verif|
+--------------------+----------------+------------+
|0005f50442cb953dc...|           53.99|       false|
|00061f2a7bc09da83...|           59.99|       false|
|00063b381e2406b52...|            45.0|       false|
|000f25f4d72195062...|          119.99|       false|
|001021efaa8636c29...|            49.0|       false|
|0011d82c4b53e22e8...|           289.0|       false|
|00130c0eee84a3d90...|            27.9|       false|
|0017afd5076e074a4...|           809.1|        true|
|001dbc16dc51075e9...|            69.9|       false|
|00276d5c3491fbf55...|            44.9|       false|
|002f19a65a2ddd70a...|           53.98|       false|
|003423b755b562962...|          232.75|       false|
|003a94f778ef8cfd5...|            39.9|       false|
|004345d16a1ab2c21...|            37.9|       false|
|0046e1d57f4c07c8c...|            7.79|       false|
|004eab0fd8f28adaf...|            21.5|       

In [ ]:
# Valeur de l'écart-type
ecartType_montant

210.64514531166327

In [ ]:
# Commandes au-dessus du seuil, de la plus élevée à la plus faible
commandes_extremes = table_mt_ecartType.filter(col("calcul_verif")).orderBy(col("montant_commande").desc())
commandes_extremes.show()

+--------------------+----------------+------------+
|            order_id|montant_commande|calcul_verif|
+--------------------+----------------+------------+
|03caa2c082116e1d3...|         13440.0|        true|
|736e1922ae60d0d6a...|          7160.0|        true|
|0812eb902a67711a1...|          6735.0|        true|
|fefacc66af859508b...|          6729.0|        true|
|f5136e38d1a14a4db...|          6499.0|        true|
|2cc9089445046817a...|          5934.6|        true|
|a96610ab360d42a2e...|          4799.0|        true|
|199af31afc78c699f...|          4690.0|        true|
|b4c4b76c642808cbe...|          4599.9|        true|
|8dbc85d1447242f3b...|          4590.0|        true|
|d2f270487125ddc41...|          4400.0|        true|
|426a9742b533fc6fe...|         4399.87|        true|
|68101694e5c5dc733...|         4099.99|        true|
|b239ca7cd485940b3...|          4059.0|        true|
|86c4eab1571921a6a...|          3999.9|        true|
|80dfedb6d17bf2353...|          3999.0|       

### Q7 — Produits d'une commande regroupés dans un Array

In [ ]:
# collect_list : liste des product_id de chaque commande (doublons conservés)
table_product = table_fait.groupBy("order_id").agg(collect_list("product_id").alias("produits"))
table_product.show()


+--------------------+--------------------+
|            order_id|            produits|
+--------------------+--------------------+
|0005f50442cb953dc...|[4535b0e1091c278d...|
|00061f2a7bc09da83...|[d63c1011f49d98b9...|
|00063b381e2406b52...|[f177554ea93259a5...|
|000f25f4d72195062...|[1c05e0964302b6cf...|
|001021efaa8636c29...|[5d7c23067ed3fc8c...|
|0011d82c4b53e22e8...|[c389f712c4b4510b...|
|00130c0eee84a3d90...|[89321f94e35fc6d7...|
|0017afd5076e074a4...|[fe59a1e006df3ac4...|
|001dbc16dc51075e9...|[777d2e438a1b645f...|
|00276d5c3491fbf55...|[71c89e9d8fa274b0...|
|002f19a65a2ddd70a...|[9eae06d51aaa383b...|
|003423b755b562962...|[84904413b82eee33...|
|003a94f778ef8cfd5...|[452f66a0f164cac5...|
|004345d16a1ab2c21...|[d0fb1e667e989933...|
|0046e1d57f4c07c8c...|[ff6caf9340512b8b...|
|004eab0fd8f28adaf...|[0c753fe6a58dfa7a...|
|00526a9d4ebde463b...|[0c4a0f8ab44f9acd...|
|005e5166e99d1e4d0...|[03bb24d19ea7449c...|
|0068c109948b9a1df...|[525f8fbde09c9bf7...|
|006e05e81c068ddc3...|[02475368d

In [ ]:
# Contrôles : une ligne par commande, aucun article perdu.
# Les 775 tableaux vides correspondent aux commandes sans article.
print("Lignes du résultat :", table_product.count())
print("Commandes distinctes dans table_fait :", table_fait.select("order_id").distinct().count())

print("Somme des tailles :", table_product.agg(sum(size("produits"))).first()[0])
print("Articles dans table_fait :", table_fait.filter(col("product_id").isNotNull()).count())

print("Tableaux vides :", table_product.filter(size("produits") == 0).count())
print("Taille max :", table_product.agg(max(size("produits"))).first()[0])

table_product.printSchema()
table_product.show(5, truncate=False)

Lignes du résultat : 99441
Commandes distinctes dans table_fait : 99441
Somme des tailles : 112650
Articles dans table_fait : 112650
Tableaux vides : 775
Taille max : 21
root
 |-- order_id: string (nullable = true)
 |-- produits: array (nullable = false)
 |    |-- element: string (containsNull = false)

+--------------------------------+----------------------------------+
|order_id                        |produits                          |
+--------------------------------+----------------------------------+
|0005f50442cb953dcd1d21e1fb923495|[4535b0e1091c278dfd193e5a1d63b39f]|
|00061f2a7bc09da83e415a52dc8a4af1|[d63c1011f49d98b976c352955b1c4bea]|
|00063b381e2406b52ad429470734ebd5|[f177554ea93259a5b282f24e33f65ab6]|
|000f25f4d72195062c040b12dce9a18a|[1c05e0964302b6cf68ca0d15f326c6ba]|
|001021efaa8636c29475e7734483457d|[5d7c23067ed3fc8c6e699b9373d5890b]|
+--------------------------------+----------------------------------+
only showing top 5 rows


### Q8 — Export du rapport final en JSON compressé, partitionné par année

In [ ]:
# Ajout de l'année d'achat puis export JSON compressé en gzip,
# avec un sous-dossier par année (annee=2016, 2017, 2018).
rapport_final = table_fait.withColumn('annee',extract(lit('YEAR'), 'order_purchase_timestamp'))


rapport_final.write.mode('overwrite').partitionBy('annee').option("compression", "gzip").json('output/rapport_json')

In [ ]:
# Vérification de l'export : structure des dossiers, extension .json.gz,
# nombre de lignes relues identique et colonne annee restituée.
# Le JSON ne conserve pas les types timestamp : ils sont relus en string.
import os

print(sorted(os.listdir("output/rapport_json")))                  # annee=2016, annee=2017, annee=2018, _SUCCESS
print(os.listdir("output/rapport_json/annee=2017")[:3])           # des fichiers en .json.gz

relu = spark.read.json("output/rapport_json")
print("lignes rapport_final :", rapport_final.count(), "| lignes relues :", relu.count())
print("annee dans les colonnes :", "annee" in relu.columns)
relu.printSchema()

['._SUCCESS.crc', '_SUCCESS', 'annee=2016', 'annee=2017', 'annee=2018']
['.part-00003-bd14cbd5-7e92-4ac7-adbf-f0a109d675ff.c000.json.gz.crc', '.part-00004-bd14cbd5-7e92-4ac7-adbf-f0a109d675ff.c000.json.gz.crc', 'part-00000-bd14cbd5-7e92-4ac7-adbf-f0a109d675ff.c000.json.gz']
lignes rapport_final : 113425 | lignes relues : 113425
annee dans les colonnes : True
root
 |-- customer_city: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: long (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_item_id: long (nullable = true)
 |-- order_purchase_timestamp: string

---
## Synthèse

Quelques résultats marquants issus des exercices :

- **99 441 commandes** pour **96 096 clients réels** : `customer_id` identifie une commande, pas un client.
- **97 %** des commandes sont livrées ; parmi celles qui ont une date de livraison, **environ 8 %** arrivent en retard.
- Le **taux de réachat est faible** : seulement **3,1 %** des clients passent une deuxième commande, en moyenne 80 jours après la première.
- La règle de Pareto ne s'applique pas pleinement : **45 % des clients** génèrent 80 % du CA.
- Le volume de commandes est multiplié par environ 9 entre janvier 2017 et janvier 2018, avec un pic en **novembre 2017** (Black Friday).
- La carte de crédit domine largement les paiements, devant le boleto.

**Points techniques abordés :** fonctions de fenêtre (`row_number`, `rank`, `lag`, cumul),
`collect_list` / `explode`, pivot, broadcast join et lecture du plan d'exécution,
`persist`, écriture Parquet et JSON partitionnée, contrôles systématiques des résultats.